<a href="https://colab.research.google.com/github/dineshaiacademy/5-day-ai-bootcamp/blob/main/Day%201%20-%20LLM%20Fundamentals/Learning/4%20-%20Getting%20Reliable%20AI%20Responses.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎯 Getting Reliable AI Responses

An LLM doesn't "know" an answer the way a calculator knows `2 + 2 = 4` — it **predicts** the most likely next words from patterns learned in training. Usually right, not always, and not always the *same* every time. This notebook is a toolbox of concrete habits — one runnable demo each — that make AI answers more consistent and trustworthy.

> ⚠️ **Runs locally against LM Studio** (`http://localhost:1234/v1`, same setup as notebooks 2–4) — won't run unmodified in Colab.

**You'll do in code:** watch temperature control randomness, chain small checkable steps instead of one big ask, get the model to check its own work, spell out what "good" means, ground it in real documents, let it say "I don't know," force a fixed JSON output, refine a draft, and test a prompt for consistency across runs.

## 📖 Why does AI sometimes give different answers?

A calculator is deterministic — same input, same output, always. A language model is a **probability distribution over the next token**, conditioned on the prompt so far. Generating text means repeatedly *sampling* from that distribution, which by construction can pick a different (still plausible) token on a different run — especially when two candidates are close in probability. True even for facts or arithmetic the model "knows" well. The rest of this notebook narrows that variability where it matters, and catches it where you can't remove it.

## ✅ Prerequisites

- [LM Studio](https://lmstudio.ai/) installed, with a chat model downloaded and its local server started (see notebook 2, Step 1, if you need a refresher)
- Python 3.9+ with your bootcamp `venv` activated

## ⚙️ Setup — Connect to Your Local Model

Same boilerplate as notebooks 2 and 3: install the SDK, point it at LM Studio's local server, and auto-detect whichever chat model is loaded.

In [ ]:
%pip install -q openai

: 

In [ ]:
from openai import OpenAI

BASE_URL = "http://localhost:1234/v1"
client = OpenAI(base_url=BASE_URL, api_key="lm-studio")  # key is required by the SDK but ignored by LM Studio

models = client.models.list()
chat_models = [m.id for m in models.data if "embed" not in m.id.lower()]
if not chat_models:
    raise RuntimeError("No chat model found. Load one in LM Studio's Developer tab and start the server.")

MODEL = chat_models[0]
print(f"✅ Using MODEL: {MODEL}")

In [ ]:
def ask(prompt, system=None, max_tokens=200, temperature=0.0):
    """Send one prompt to the local model and print the reply. Used throughout this notebook."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(
        model=MODEL, messages=messages, max_tokens=max_tokens, temperature=temperature,
    )
    print(response.choices[0].message.content.strip())

## 🎲 Make AI Less Random with Temperature

`temperature` reshapes the next-token probability distribution before sampling. Near `0`, it sharpens toward the single highest-probability token — close to deterministic, good for facts, arithmetic, code, where there's one correct answer. Near `1` (or higher), it flattens the distribution so lower-probability tokens get picked more often — more varied and "creative," good for brainstorming or fiction. Temperature doesn't make the model smarter or dumber, only more or less willing to gamble on a less-likely word.

## 🧪 Demo — Same Prompt, Different Temperatures

Same factual prompt, three runs at `temperature=0` (should look nearly identical) and three at `temperature=1.0` (expect variety in wording, occasionally in content).

In [ ]:
question = "In one sentence, what is the capital of Australia and why was it chosen over Sydney or Melbourne?"

print("🌡️ LOW temperature (0.0) — run 3 times")
print("-" * 40)
for i in range(3):
    print(f"Run {i+1}:")
    ask(question, temperature=0.0, max_tokens=60)

print("\n🌡️ HIGH temperature (1.0) — run 3 times")
print("-" * 40)
for i in range(3):
    print(f"Run {i+1}:")
    ask(question, temperature=1.0, max_tokens=60)

## 🧩 Turn a Big Job into Small Jobs

One large, multi-part request forces the model to split limited attention across every sub-task at once, and makes one wrong sub-answer hard to isolate — you get one long response to audit instead of several short ones. **Decomposition** — splitting a task into a sequence of smaller prompts, optionally feeding each output into the next as context — trades a little orchestration for: each step being individually verifiable, each step getting full attention, and a wrong step being fixable without redoing the whole chain. Most production LLM systems are chains of small, narrow prompts rather than one giant one.

## 🧪 Demo — One Big Ask vs. a Small Chain

The *same* task (analyze a contract clause), first as one big request, then as a 3-step chain where each answer is easy to check on its own.

In [ ]:
clause = (
    "\"Either party may terminate this agreement with 30 days written notice. "
    "The Client agrees to pay all fees accrued through the termination date, plus a 10% early-termination fee "
    "if terminated before the 12-month mark. Late payments accrue 1.5% interest per month.\""
)

print("❌ ONE BIG REQUEST")
print("-" * 40)
ask(f"Read this contract clause and tell me if it is safe:\n{clause}", max_tokens=250)

In [ ]:
print("✅ SMALL CHAIN — Step 1: payment rules")
print("-" * 40)
ask(f"List only the payment-related rules in this clause, as short bullet points:\n{clause}", max_tokens=120)

print("\n✅ SMALL CHAIN — Step 2: termination rules")
print("-" * 40)
ask(f"List only the termination-related rules in this clause, as short bullet points:\n{clause}", max_tokens=120)

print("\n✅ SMALL CHAIN — Step 3: biggest risk")
print("-" * 40)
ask(f"In one sentence, what is the single biggest financial risk to the Client in this clause?\n{clause}", max_tokens=80)

## 🔍 Ask AI to Check Its Own Answer

Asking the model to re-derive or re-check its own answer in the same response is **self-verification**: it produces a result, then audits that result rather than just restating it. This catches a real but limited class of errors (arithmetic slips, dropped constraints) because re-deriving is a different generation path than the first pass. It does **not** catch errors the model is systematically confident about — it will just confidently confirm the same wrong reasoning — and it's no substitute for an independent check (calculator, unit test, second source) when correctness actually matters.

In [ ]:
ask(
    "Calculate 18% of $142. Then, on a new line, check your calculation again from scratch. "
    "If you find a mistake, correct it and give the final answer.",
    max_tokens=200,
)

## 🎯 Tell AI What "Good" Means

"Good" isn't a property the model can look up — it's a placeholder for constraints you haven't stated (length, tone, what to include/avoid, how to end). Every constraint left implicit gets filled from the training distribution's *average* case, rarely what you wanted. Same idea as the RTCF framework (notebook 4), applied specifically to the word "good": replace it with concrete, checkable rules.

In [ ]:
print("❌ Undefined \"good\"")
print("-" * 40)
ask("Write a good product description for a stainless steel water bottle.", max_tokens=150)

print("\n✅ \"Good\" spelled out as rules")
print("-" * 40)
ask(
    "Write a product description for a stainless steel water bottle. "
    "Rules: under 50 words, must include the price ($24), do not use the words 'best' or 'amazing', "
    "end the description with exactly 'Buy now.'",
    max_tokens=150,
)

## 📚 Give AI the Information It Needs

A model's training data is a frozen, general snapshot — it doesn't contain your company's internal policy or today's data, no matter how confidently it answers. Pasting the real source document into the prompt (**grounding**) means the model completes text *conditioned on the facts you supplied* instead of falling back on pretrained generalities. This is the core idea behind **RAG (Retrieval-Augmented Generation)**: retrieve the relevant documents, then put them in the prompt alongside the question, so the model answers from your real data instead of guessing. We build this properly on Day 2 — for now: if you have the real information, give it to the model.

In [ ]:
policy = (
    "Refund Policy: Items may be returned within 14 days of delivery for a full refund, provided they are unused "
    "and in original packaging. Sale items are final sale and cannot be refunded. Shipping costs are non-refundable."
)

print("❌ No information given — model has to guess")
print("-" * 40)
ask("What is our company\'s refund policy?", max_tokens=120)

print("\n✅ Real document given — model is grounded")
print("-" * 40)
ask(f"Here is our refund policy:\n{policy}\n\nIn 2 sentences, explain the refund rules to a customer.", max_tokens=120)

## 🤷 Let AI Say "I Don't Know"

By default a model is trained to produce a plausible-looking completion for almost any prompt, with no built-in reflex to refuse when it lacks the fact — so it can fabricate a fluent, confident answer (a **hallucination**) indistinguishable from a correct one. Explicitly stating that "I don't know" is an acceptable answer makes refusal a valid output instead of a failure to complete the task. Not a guarantee against hallucination — the model has no true self-awareness of what it knows — but it measurably reduces confident fabrication, for nearly free.

In [ ]:
print("❌ No permission to say \'I don\'t know\'")
print("-" * 40)
ask("What was the exact attendance figure at the 2024 Dinesh AI Academy graduation ceremony?", max_tokens=100)

print("\n✅ Explicit permission to decline")
print("-" * 40)
ask(
    "What was the exact attendance figure at the 2024 Dinesh AI Academy graduation ceremony? "
    "If you don't know the answer, say 'I don't know.' Do not guess.",
    max_tokens=100,
)

## 📦 Use a Fixed Output Format

If another program will read the model's output, free-form prose is unreliable to parse — field order and wording can shift between runs. Instructing the model to respond in a fixed schema (usually **JSON**) constrains the output structurally, the same way a format instruction constrains length. This is prompt-level formatting, not a hard guarantee — for production systems that must get valid JSON every time, most providers also offer a dedicated "structured output"/"JSON mode" parameter enforced server-side; the prompt-only version here is the portable one that works with any model.

In [ ]:
ask(
    "Give me information about a product: a stainless steel water bottle, $24, currently in stock. "
    "Respond with ONLY valid JSON, no explanation, no markdown code fences, using exactly these keys: "
    "name, price, in_stock.",
    max_tokens=100,
)

## 🔄 Don't Expect the First Answer to Be Perfect

Since generation is one pass conditioned on everything said so far, the fastest way to fix a specific flaw isn't to re-ask from scratch — it's to keep the flawed answer in the conversation and issue a **targeted follow-up** naming exactly what's wrong. The model then conditions its next completion on both the original context *and* the correction, a much narrower task than "try again." This generate → critique → revise loop is the same pattern behind automated self-improvement, where the "critique" step can itself be another prompt.

In [ ]:
article = (
    "The company\'s Q3 revenue grew due to strong demand in the Asia-Pacific region, driven mainly by increased "
    "sales of its flagship subscription product following a price adjustment in July."
)

print("Step 1: first draft")
print("-" * 40)
ask(f"Summarize this in exactly 3 bullet points:\n{article}", max_tokens=150)

print("\nStep 2: targeted refinement")
print("-" * 40)
ask(
    f"Summarize this in exactly 3 bullet points:\n{article}\n\n"
    "Now make the region point more specific: name the region explicitly instead of saying 'the region'.",
    max_tokens=150,
)

## 🔁 Test the Same Prompt Several Times

Because a model samples from a distribution, one run tells you almost nothing about how **reliable** a prompt is — just one draw. Running the same prompt several times (at the temperature you actually intend to use) and comparing outputs is a quick way to estimate variance before relying on a prompt for anything important: if runs mostly agree, it's stable; if they diverge in structure, facts, or format, tighten the prompt (lower temperature, stricter format, more context) rather than just re-running and hoping.

In [ ]:
prompt = "List the top 3 benefits of daily exercise, as a numbered list, each under 8 words."

print("Running the same prompt 4 times at temperature=0.9:")
print("-" * 40)
for i in range(4):
    print(f"\nRun {i+1}:")
    ask(prompt, temperature=0.9, max_tokens=80)

## 🎯 The Big Idea

Don't assume "AI gave me an answer, so it must be correct." Instead: match **temperature** to the task, **break big jobs** into checkable steps, ask it to **check its own work**, say exactly what **"good"** means, **give it real information** instead of letting it guess, let it say **"I don't know,"** ask for a **fixed format** when a program will read the answer, treat the first answer as a **draft**, and **test the same prompt** several times before trusting it.

## 📝 Recap

| Concept | What you learned |
|---|---|
| Why answers vary | The model samples the next token rather than looking up a fixed fact |
| Temperature | Low = focused/predictable (facts, code, math); high = varied/creative |
| Decomposition | Split one hard-to-check request into a chain of small, verifiable ones |
| Self-verification | Re-derive-and-check catches some errors, not systematic ones |
| Defining "good" | Replace vague quality words with concrete, checkable rules |
| Grounding / RAG | Give the model the real source document instead of letting it guess |
| Permission to decline | Explicitly allowing "I don't know" reduces confident fabrication |
| Fixed format | Constrain output structure (e.g. JSON) when a program reads the answer |
| Iterative refinement | Treat the first answer as a draft; issue targeted follow-ups |
| Consistency testing | Run a prompt several times and compare before trusting it |

**Next:** `llm_fundamentals_multi_provider.ipynb` — these techniques together with RTCF, inside a real multi-turn conversation.

## 🏋️ Try It Yourself (optional)

Using the `ask()` helper defined above:

1. Pick a factual question and run it 5 times at `temperature=0.0`, then 5 times at `temperature=1.0`. Compare how much the answers vary.
2. Take one long, multi-part request you'd normally ask in one go, and split it into a 3-step chain of small prompts instead. Compare how much easier the small answers are to check.
3. Write a prompt that asks for a JSON object describing a book (title, author, year, in_stock). Run it 3 times and check whether the JSON is valid and uses the same keys every time.